## RAG Evelution Using Faiss 

In [ ]:
# Load Data 
import warnings 
warnings.filterwarnings('ignore')
from langchain_community.document_loaders import PyPDFLoader 
loader = PyPDFLoader('Static GK 2025.pdf')
pages = loader.load()

In [2]:
# Prepare and split data 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
import hashlib

spliter = RecursiveCharacterTextSplitter(chunk_size=1400 , chunk_overlap=180)
text_spliter = spliter.split_documents(pages)
chunks = [i.page_content for i in text_spliter]
metadata = [i.metadata for i in text_spliter]
ids = [hashlib.md5(chunk.encode('utf-8')).hexdigest() for chunk in chunks]
print(f'print first 5 ids : {ids[:2]}')

print first 5 ids : ['df52eef7bfa55759b4642211e13e3020', '622d6c3b19974d6f39f9950848df1607']


In [3]:
# Create Embedding 
from sentence_transformers import SentenceTransformer 
embedding = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2")
encode_chunks = embedding.encode(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
dimension = encode_chunks.shape[1]
print(f'the dimension is : {dimension}')

the dimension is : 384


In [5]:
# Using Indexing search 
import faiss  
indexing = faiss.IndexFlatL2(dimension)
indexing.add(encode_chunks)
print(f'sucessfully : {indexing}')

sucessfully : <faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x128f16870> >


In [6]:
# LLM call 
import os 
from dotenv import load_dotenv 
from langchain_groq import ChatGroq 
load_dotenv()
try:
    key = os.getenv('GROQ_API_KEY')
    print(bool(key))
except Exception as e:
    print(str(e))
    
Groq = ChatGroq(model="openai/gpt-oss-120b")

True


In [7]:
# Retrive Data 
def Retrive_Data(query:str):
    prompt = f""" write the query based on symentic search : {query} """
    query_re = Groq.invoke(prompt).content
    query_embedding = embedding.encode([query_re])
    dis , docs = indexing.search(x=query_embedding,k=3)
    threshold = 1.5
    print(f'the distance is : {dis[0]}')
    dense_docs = []
    for i , d in zip(dis[0],docs[0]):
        if threshold > i :
            dense_docs.append(chunks[d])
    return dense_docs

def Generate_answer(question:str , contexts_list : list ):
    if not contexts_list:
        return "NOT RELATED CONTENT"
    content_string = "\n\n".join(contexts_list)
    
    question = "what is the largest country in the world? "
    prompt = f""" You are a AI assistent so provided user's asking questions answers based on 
    local given document .
    content : {content_string}
    question : {question}
    """
    # Generate 
    result = Groq.invoke(prompt)
    return result.content

In [8]:
!uv pip install ragas datasets

Using Python 3.12.13 environment at: /Users/debajyotihazra/Documents/MultiAgentic RAG /.venv
Checked 2 packages in 64ms


In [9]:
!uv pip install "langchain-community<0.4"

Using Python 3.12.13 environment at: /Users/debajyotihazra/Documents/MultiAgentic RAG /.venv
Checked 1 package in 4ms


In [10]:
from datasets import Dataset 
from ragas import evaluate 
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from langchain_community.embeddings import HuggingFaceEmbeddings 

qus = "what is the largest country in the world?"
Ground_truth = "the largest country in the world by area is **Russia**"

retrive_list =  Retrive_Data(query=qus)
final_answer = Generate_answer(question=qus , contexts_list=retrive_list)

data = {
    "user_input": [qus],              
    "response": [final_answer],        
    "retrieved_contexts": [retrive_list], 
    "reference": [Ground_truth]            
}
dataset = Dataset.from_dict(data)

ragas_embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") 
result = evaluate(
    dataset=dataset,
    metrics=[Faithfulness(), AnswerRelevancy(), ContextPrecision(), ContextRecall()],
    llm=Groq,
    embeddings=ragas_embedding

)
print(result)

the distance is : [1.1781712 1.2609403 1.2962244]


/var/folders/j2/t5w7lcnj6wg9gmp3h6zxzlk00000gn/T/ipykernel_27069/1866556763.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  ragas_embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'faithfulness': 1.0000, 'answer_relevancy': 0.8419, 'context_precision': 1.0000, 'context_recall': 1.0000}
